Здесь указать Фамилию, имя, группу студента.
Хорошо бы заменить в названии файла Empty на транслитерацию фамилии ;) .

# Задание 7. Применение сверточной нейронной сети для классификации изображений
В работе используется датасет CIFAR-100.
##### Индивидуальное задание: 
**Сюда скопировать ваше индивидуальное задание**

In [ ]:
# КОНСТАНТЫ
# Если вы будете использовать приведенные в ноутбуке файлы для работы с датасетом, 
# присвойте переменной my_class номер вашего варианта
my_class = 1


## Порядок работы:  
### 1. Подключение библиотек

In [ ]:
# Код приведен в качестве примера. Здесь может быть ваш код

import numpy as np # библиотека для работы с массивами
import pandas as pd # библиотека для численного анализа и работы с датафреймами

import matplotlib.pyplot as plt # библиотека для визуализации
import seaborn as sns # библиотека для более красивой визуализации

from sklearn.model_selection import train_test_split # метод для разбиения датасета
from sklearn.metrics import confusion_matrix # построение матрицы ошибок

import tensorflow as tf
from tensorflow.keras import utils # утилиты
from tensorflow.keras.models import Sequential # модель последовательной нейросети
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout # слои нейросети - могут потребоваться и другие

from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau # это новый инструмент, см. ниже

# фиксируем последовательность генератора случайных чисел для поворяемости результатов
SEED = 42
from tensorflow.random import set_seed
from numpy.random import seed
seed(SEED)
set_seed(SEED)

In [ ]:
# функция для работы с метаданными и метками классов
def unpickle(file):
    import pickle
    with open(file, 'rb') as fo:
        dict = pickle.load(fo, encoding='bytes')
    return dict

### 2. Загрузка датасета

#### 2.1 Загружаем датасет CIFAR-100.  
Датасет уже разделен на обучающую и тестовую выборки.  
После загрузки датасета проверим размерность обучающей выборки и число уникальных меток классов.

In [ ]:
# Код приведен в качестве примера. Здесь может быть ваш код

from tensorflow.keras.datasets import cifar100
(xf_train, yf_train), (xf_test, yf_test) = cifar100.load_data(label_mode='fine') # загружаем датасет с разметкой на классы
(xc_train, yc_train), (xc_test, yc_test) = cifar100.load_data(label_mode='coarse') # загружаем датасет с разметкой на суперклассы

print("Размерность обучающей выборки: ", xf_test.shape, "Размерность вектора меток классов: ", yf_test.shape)

num_classes = len(np.unique(yf_train))
num_superclasses = len(np.unique(yc_train))

print ("Число уникальных меток супер-классов: ", num_superclasses)
print ("Число уникальных меток классов: ", num_classes)

#### 2.2 Считываем текстовые метки классов и суперклассов

In [ ]:
metadata_path = 'meta' # актуализировать путь к файлу с метаданными
metadata = unpickle(metadata_path)
superclass_dict = dict(list(enumerate(metadata[b'coarse_label_names'])))
print("Список названий суперклассов: \n", superclass_dict)

In [ ]:
classes = dict(list(enumerate(metadata[b'fine_label_names'])))
print("Список названий первых пяти классов:")
for i in range (5): print(classes[i])

**Выполним проверочный вывод изображений из датасета:**

In [ ]:
plt.figure(figsize=(10,3))
for i in range(100,112):
    plt.subplot(2,6,i-100+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(xf_train[i])
    plt.xlabel(classes[yf_train[i][0]])

### 3. Многоклассовая классификация изображений: подготовка данных для обучения и тестирования модели

#### 3.1. Выбираем из общей обучающей выборки суперкласс согласно варианту
В варианте задания указан номер суперкласса.  
Для дальнейшей работы его нужно выполнить следующие действия: 
1. Выделить из исходной обучающей и тестовой выборки и векторов меток элементы указанного суперкласса и его меток. (рекомендуется сохранить их в новых датафреймах). Выделение объектов указанного суперкласса рекомендуется оформить в виде функции, т.к. оно будет использоваться несколько раз.
2. Сохранить полученные данные в датафреймах соотвествующей размерности.

In [ ]:
# Выделяем объекты класса из обучающей выборки

# Код приведен в качестве примера (но не примера для подражания :) ). Здесь может быть ваш код

x_train = np.empty(0,int)
y_train = np.empty(0,int)

#for i in (0,len(y_train)-1,1):
for i in range(0, len(yf_train)):
    if (yc_train[i] == my_class):
        x_train = np.append(x_train, (xf_train[i]))
        y_train = np.append(y_train, yf_train[i])   

y_train = np.reshape(y_train,(len(y_train), 1))
x_train = np.reshape(x_train, (len(y_train),32,32,3))
print ("Размерность датафрейма обучающей выборки: ", x_train.shape)
print ("Размерность датафрейма меток обучающей выборки: ", y_train.shape)

**При желании можно сделать проверочный вывод и убедиться, что обучающая выборка сформирована корректно**

In [ ]:
# проверочный вывод
plt.figure(figsize=(10,3))
for i in range(1,13):
    plt.subplot(2,6,i)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(x_train[i])
    plt.xlabel(classes[y_train[i][0]])

#### 3.2. Выбираем из общей тестовой выборки суперкласс согласно варианту
Выполняем те же действия с тестовой выборкой

In [ ]:
# Выделяем объекты класса из тестовой выборки

# Код приведен в качестве примера (но не примера для подражания :) ). Здесь может быть ваш код

x_test = np.empty(0,int)
y_test = np.empty(0,int)

#for i in (0,len(y_train)-1,1):
for i in range(0, len(yf_test)):
    if (yc_test[i] == my_class):
        x_test = np.append(x_test, (xf_test[i]))
        y_test = np.append(y_test, yf_test[i])   

y_test = np.reshape(y_test,(len(y_test), 1))
x_test = np.reshape(x_test, (len(y_test),32,32,3))
print ("Размерность датафрейма тестовой выборки: ", x_test.shape)
print ("Размерность датафрейма меток тестовой выборки: ", y_test.shape)

In [ ]:
plt.figure(figsize=(10,3))
for i in range(1,13):
    plt.subplot(2,6,i)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(x_test[i])
    plt.xlabel(classes[y_test[i][0]])

#### 3.3. Нормализация данных и кодирование меток

Для окончательной подготовки данных необходимо:
1. выполнить нормализацию: цвет пикселя цветного изображения кодируется тремя целыми числами, значение которых лежит в интервале [0,255]. Рекомендуется привести их к интервалу [0,1].
2. Перекодировать метки классов: в суперклассе содержатся представители 5 классов, и их метки могут быть пронумерованы произвольно, а для обучения их нужно перенумеровать по порядку.
3. Выполнить преобразование меток класса в формат one hot encoding.

***Подсказка:*** можно использовать функцию `to_categorical` модуля `utils` библиотеки `tensorflow.keras`

In [ ]:
# нормализуем исходные данные

# Здесь должен быть ваш код


In [ ]:
# Перекодируем метки обучающей выборки

# Здесь должен быть ваш код


In [ ]:
# Перекодируем метки тестовой выборки


In [ ]:
# Кодируем метки one-hot

# Здесь должен быть ваш код

y_train[:5]

### 4. Описание модели нейронной сети на базе класса Sequential 

Описание нейронной сети выполнить любым удобным способом.  
Архитектура сети должна соответствовать индивидуальному заданию.  
***Напоминание:***  
1. Слой входных сигналов нейронной сети должен соответствовать описанию примера обучающей выборки.  
2. Для многоклассовой классификации выходной слой содержит число нейронов, равное числу классов.  
3. Рекомендуемая функция активации выходного слоя - `softmax`.
4. Не забываем, что функция `softmax` возвращает не номер класса, а вероятность принадлежности объекта классу.  

In [ ]:
# Для удобства проверки просьба вставить здесь описание нейросети из задания
# I→ C2D(16(3*3),relu)→ MP(2*2)→ C2D(16(3*3),relu)→ MP(2*2)→ F→ D(32,relu)→ D(out)

# Здесь должен быть ваш код


#### 4.3. Компиляция модели нейронной сети с соответствующими параметрами

Выполнить компиляцию модели с указанием актуальных параметров.  
Рекомендуется ознакомиться с документацией и примерами и разумно указать (как минимум) параметры loss, optimizer, metrics.  
Вывести описание (summary) модели и при желании - ее визуализацию.  
***Подсказка:*** для мультиклассовой классификации рекомендуется использовать функцию ошибки `categorical_crossentropy`

In [ ]:
# Здесь должен быть ваш код


### 5. Обучение модели

Если вы уже обучали модели нейронных сетей, то у вас скорее всего уже возникало желание оптимизировать и автоматизировать некоторые моменты этого процесса.  
В Keras для этого используется механизм обратных вызовов - `Callbacks`.  
`Callbacks` в Keras — это объекты, которые позволяют настроить поведение модели во время обучения.  
Некоторые возможности Callbacks:  
- выполнение валидации в различных точках во время обучения;  
- установление контрольных точек модели через регулярные интервалы или когда она превышает определённый порог точности;   
- изменение скорости обучения модели, когда обучение перестаёт сходиться;   
- тонкая настройка верхних слоёв, когда обучение перестаёт сходиться;   
- и даже отправка электронных писем или сообщений, когда обучение заканчивается или когда превышен определённый порог производительности и т. д.   
 Более подробную информацию можно найти в [документации TensorFlow / Keras ](https://www.tensorflow.org/api_docs/python/tf/keras/callbacks).  
Пример описания `Callbacks` с настройками, близкими к настройкам "по умолчанию", приведен ниже.  

In [ ]:
# Код приведен в качестве примера. Здесь может быть ваш код

reduce_lr = ReduceLROnPlateau(
    monitor='val_accuracy', 
    factor=0.5,
    patience=5, 
    min_lr=0.000001,
    verbose=0)

save_best = ModelCheckpoint(
    filepath="base_weights.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="auto",
    verbose=1)

early_stop = EarlyStopping(
    monitor="val_accuracy",
    min_delta=0.0001,
    patience=5,
    verbose=1,
    mode='auto',
    baseline=None,
    restore_best_weights=True,
    start_from_epoch=0
)

Ниже приведен пример обучения модели с использованием `Callbacks`.  

In [ ]:
# Код приведен в качестве примера. Здесь может быть ваш код

# Обучаем модель
history = baseline_model.fit(x_train, y_train,
              batch_size=32,
              epochs=25,
              validation_data=(x_test, y_test),
              callbacks=[save_best, reduce_lr, early_stop],
              verbose=1)

Вывести графики, демонстрирующие динамику процесса обучения.

In [ ]:
# Здесь должен быть ваш код


### 6. Оценка качества работы многоклассового классификатора  

#### 6.1 Определить долю правильных ответов на обучающей и тестовой выборке

In [ ]:
# Здесь должен быть ваш код


#### 6.2. Построить матрицу ошибок

In [ ]:
# Здесь должен быть ваш код


### 7. Оптимизация нейронной сети

Используя знания, полученные при изучении основного и дополнительного материала, внести в исходную `baseline` модель изменения таким образом, чтобы повысить качество работы.  
**Целевое значение accuracy не менее 0,7**  
В отчете сохранить результаты всех проведенных экспериментов с выводами и обосновать выбор лучшего результата.  
Требуется продемонстрировать не менее 3 вариантов модификаций `baseline` модели. Попытаться обосновать ваши действия. 

In [ ]:
# При необходимости импортируем нужные модули

# Здесь может быть ваш код


Описываем на основе `baseline` оптимизированную модель, обучаем, тестируем полученный результат, делаем вывод.

In [ ]:
# Вносим изменения в baseline-модель 

# Здесь должен быть ваш код

# Компилируем оптимизированную модель 

# Здесь должен быть ваш код

# Выводим описание оптимизированной модели 
# Здесь должен быть ваш код


**Подсказка:** Для повышения качества обученной модели можно экспериментировать не только с параметрами самой модели, но и с параметрами процесса обучения за счет настройки Callbacks.

In [ ]:
# Настраиваем Callbacks

# Здесь может быть ваш код


In [ ]:
# Обучаем модель
# Здесь должен быть ваш код

# Визуализируем динамику процесса обучения
# Здесь должен быть ваш код


In [ ]:
# Определяем accuracy для обучающей и тестовой выборок
# Здесь должен быть ваш код

# Строим матрицу ошибок, определяем метрику качества
# Здесь должен быть ваш код



## 8. Сохранение оптимизированной модели в файл

In [ ]:
# Здесь должен быть ваш код


## ВЫВОДЫ ПО ПРОДЕЛАННОЙ РАБОТЕ

**ЗДЕСЬ ДОЛЖЕН БЫТЬ ВАШ ТЕКСТ**